In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/achiver7/notebook6cfab69cee/__results__.html
/kaggle/input/notebooks/achiver7/notebook6cfab69cee/__notebook__.ipynb
/kaggle/input/notebooks/achiver7/notebook6cfab69cee/__output__.json
/kaggle/input/notebooks/achiver7/notebook6cfab69cee/custom.css
/kaggle/input/notebooks/achiver7/notebook6cfab69cee/run_arch2_preln_efficient/latest.pt
/kaggle/input/notebooks/achiver7/notebook6cfab69cee/run_arch2_preln_efficient/log.jsonl
/kaggle/input/notebooks/achiver7/notebook6cfab69cee/run_arch2_preln_efficient/best.pt
/kaggle/input/datasets/achiver7/project-phase-1/data/sc_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/test.json
/kaggle/input/datasets/achiver7/project-phase-1/data/sc_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribun

In [2]:
CKPT = "/kaggle/input/notebooks/achiver7/notebook6cfab69cee/run_arch2_preln_efficient/latest.pt"

In [3]:
# ============================================================
# ARCHITECTURE 2 (PRE-LN) — DEFINITIONS ONLY (no training). Author's code, sections 0-24 unchanged
# except: the old-run folder is no longer deleted. Use for evaluation / generation.
# (original header follows)
#
#
# Uses only:
#   /kaggle/input/.../tokenizer.json
#   /kaggle/input/.../*.bin
#
# NEVER loads the old run_arch2_encdec checkpoint.
#
# Model:
#   129,957,888 parameters
#   d_model=768
#   heads=8
#   d_ff=3072
#   6 encoder + 6 decoder layers
#   Pre-LN
#   GELU
#   tied embeddings/output head
#   sinusoidal positions
#
# Efficiency:
#   fused SDPA attention
#   micro-batch=2
#   grad accumulation=8
#   fused AdamW
#   AMP fp16
#   TF32 matmul
#
# Fresh start:
#   old checkpoint directory is NOT read
#   new run directory is deleted/recreated
# ============================================================

import os
import time
import math
import json
import shutil
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer


# ============================================================
# 0. DEVICE / PERFORMANCE
# ============================================================

assert torch.cuda.is_available(), "CUDA GPU required"

DEV = torch.device("cuda")

print("CUDA:", torch.cuda.get_device_name(0))

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

try:
    torch.backends.cuda.matmul.allow_tf32 = True
except Exception:
    pass

try:
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass


# ============================================================
# 1. PATHS / CONFIG
# ============================================================

TOKD = (
    "/kaggle/input/datasets/achiver7/"
    "phase1-tokens-v40k-clean1/tokens"
)

# NEW directory: old collapsed checkpoint is never reused.
RUN = "/kaggle/working/run_arch2_preln_efficient"

os.makedirs(RUN, exist_ok=True)   # definitions-only copy: never deletes anything

V = 40000

CFG = dict(
    BLOCK=1024,

    SRC_LEN=768,
    TGT_LEN=256,

    MICRO_BATCH=2,
    GRAD_ACCUM=8,

    LR=3e-4,
    MIN_LR_FRAC=0.1,
    WD=0.1,
    WARMUP=500,

    STEPS=10000,
    CLIP=1.0,

    EVAL_EVERY=500,

    SEED=42,

    MIX=dict(
        acts=0.273,
        sc=0.469,
        tri=0.258,
    ),
)

BLOCK = CFG["BLOCK"]
S = CFG["SRC_LEN"]
TT = CFG["TGT_LEN"]
MB = CFG["MICRO_BATCH"]
GA = CFG["GRAD_ACCUM"]

assert S + TT == BLOCK
assert 50_000_000 <= 129_957_888 <= 300_000_000
assert abs(sum(CFG["MIX"].values()) - 1.0) < 1e-6

print(
    f"Config: SRC={S} "
    f"TGT={TT} "
    f"MB={MB} "
    f"GA={GA} "
    f"effective={MB*GA}"
)

print("RUN:", RUN)
print("FRESH RUN: no previous checkpoint will be loaded.")


# ============================================================
# 2. TOKENIZER
# ============================================================

tokenizer_path = f"{TOKD}/tokenizer.json"

assert os.path.exists(tokenizer_path), tokenizer_path

tok = Tokenizer.from_file(tokenizer_path)

BOS = tok.token_to_id("<bos>")

assert BOS is not None, "Tokenizer has no <bos>"

print("BOS token:", BOS)


# ============================================================
# 3. SEEDS
# ============================================================

np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])


# ============================================================
# 4. EFFICIENT ATTENTION
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        dropout=0.1
    ):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            d_model
        )

        self.k_proj = nn.Linear(
            d_model,
            d_model
        )

        self.v_proj = nn.Linear(
            d_model,
            d_model
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        query,
        key_value,
        mask=None,
        is_causal=False
    ):

        B, Tq, _ = query.shape
        _, Tk, _ = key_value.shape

        Q = self.q_proj(query)
        K = self.k_proj(key_value)
        V = self.v_proj(key_value)

        Q = Q.view(
            B,
            Tq,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        K = K.view(
            B,
            Tk,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        V = V.view(
            B,
            Tk,
            self.n_heads,
            self.d_head
        ).transpose(1, 2)

        dropout_p = (
            self.dropout.p
            if self.training
            else 0.0
        )

        if is_causal:

            # Native causal kernel.
            context = F.scaled_dot_product_attention(
                Q,
                K,
                V,
                attn_mask=None,
                dropout_p=dropout_p,
                is_causal=True,
            )

        else:

            if mask is not None:
                mask = mask.to(
                    device=Q.device,
                    dtype=Q.dtype
                )

            context = F.scaled_dot_product_attention(
                Q,
                K,
                V,
                attn_mask=mask,
                dropout_p=dropout_p,
                is_causal=False,
            )

        context = (
            context
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                Tq,
                self.d_model
            )
        )

        return self.out_proj(context)


# ============================================================
# 5. SINUSOIDAL POSITIONAL ENCODING
# ============================================================

def make_sinusoidal_table(
    max_len,
    d_model
):

    position = torch.arange(
        max_len,
        dtype=torch.float32
    ).unsqueeze(1)

    k = torch.arange(
        0,
        d_model,
        2,
        dtype=torch.float32
    )

    omega = 1.0 / (
        10000 ** (k / d_model)
    )

    table = torch.zeros(
        max_len,
        d_model,
        dtype=torch.float32
    )

    table[:, 0::2] = torch.sin(
        position * omega
    )

    table[:, 1::2] = torch.cos(
        position * omega
    )

    return table


# ============================================================
# 6. FEED-FORWARD
# ============================================================

class FeedForward(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.linear1 = nn.Linear(
            d_model,
            d_ff
        )

        self.activation = nn.GELU()

        self.linear2 = nn.Linear(
            d_ff,
            d_model
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.dropout(x)

        return x


# ============================================================
# 7. PRE-LN ENCODER LAYER
# ============================================================

class TransformerEncoderLayerPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        # Pre-LN:
        # x + Attention(LN(x))

        nx = self.norm1(x)

        x = x + self.dropout(
            self.self_attn(
                nx,
                nx
            )
        )

        # Pre-LN:
        # x + FFN(LN(x))

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# 8. PRE-LN ENCODER
# ============================================================

class TransformerEncoderPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerEncoderLayerPreLN(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        return x


# ============================================================
# 9. PRE-LN DECODER LAYER
# ============================================================

class TransformerDecoderLayerPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.cross_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.norm3 = nn.LayerNorm(
            d_model
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
        memory
    ):

        # ----------------------------------------------------
        # 1. Causal self-attention
        # ----------------------------------------------------

        nx = self.norm1(x)

        x = x + self.dropout(
            self.self_attn(
                nx,
                nx,
                is_causal=True
            )
        )

        # ----------------------------------------------------
        # 2. Cross-attention
        # Q = decoder
        # K/V = encoder
        # ----------------------------------------------------

        x = x + self.dropout(
            self.cross_attn(
                self.norm2(x),
                memory
            )
        )

        # ----------------------------------------------------
        # 3. FFN
        # ----------------------------------------------------

        x = x + self.ffn(
            self.norm3(x)
        )

        return x


# ============================================================
# 10. PRE-LN DECODER
# ============================================================

class TransformerDecoderPreLN(
    nn.Module
):

    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        max_len=256,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerDecoderLayerPreLN(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        # Kept for diagnostics/check compatibility.
        self.register_buffer(
            "causal_mask",
            torch.triu(
                torch.full(
                    (max_len, max_len),
                    float("-inf")
                ),
                diagonal=1
            ),
            persistent=False
        )

    def forward(
        self,
        x,
        memory
    ):

        for layer in self.layers:

            x = layer(
                x,
                memory
            )

        return x


# ============================================================
# 11. COMPLETE PRE-LN MODEL
# ============================================================

class TransformerModelPreLN(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        max_src_len,
        max_tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

        # Shared embedding.
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Fixed source positions.
        self.register_buffer(
            "src_pos_encoding",
            make_sinusoidal_table(
                max_src_len,
                d_model
            ),
            persistent=False
        )

        # Fixed target positions.
        self.register_buffer(
            "tgt_pos_encoding",
            make_sinusoidal_table(
                max_tgt_len,
                d_model
            ),
            persistent=False
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.encoder = TransformerEncoderPreLN(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            dropout=dropout
        )

        self.decoder = TransformerDecoderPreLN(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            max_len=max_tgt_len,
            dropout=dropout
        )

        # Tied output head.
        self.output_projection = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        self.output_projection.weight = (
            self.token_embedding.weight
        )

        self._init_weights()

    def _init_weights(self):

        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02
        )

        for module in self.modules():

            if isinstance(
                module,
                nn.Linear
            ):

                if (
                    module.weight
                    is not self.token_embedding.weight
                ):

                    nn.init.xavier_uniform_(
                        module.weight
                    )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

            elif isinstance(
                module,
                nn.LayerNorm
            ):

                nn.init.ones_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def encode(self, src):

        _, T = src.shape

        assert T <= self.max_src_len

        x = self.token_embedding(src)

        x = x * math.sqrt(
            self.d_model
        )

        x = (
            x
            + self.src_pos_encoding[
                :T
            ].unsqueeze(0)
        )

        x = self.dropout(x)

        return self.encoder(x)

    def decode(
        self,
        tgt_in,
        memory
    ):

        _, T = tgt_in.shape

        assert T <= self.max_tgt_len

        x = self.token_embedding(
            tgt_in
        )

        x = x * math.sqrt(
            self.d_model
        )

        x = (
            x
            + self.tgt_pos_encoding[
                :T
            ].unsqueeze(0)
        )

        x = self.dropout(x)

        x = self.decoder(
            x,
            memory
        )

        return self.output_projection(x)

    def forward(
        self,
        src,
        tgt_in
    ):

        memory = self.encode(src)

        return self.decode(
            tgt_in,
            memory
        )


# ============================================================
# 12. BUILD MODEL
# ============================================================

def build_model(
    vocab_size,
    src_len,
    tgt_len
):

    assert src_len + tgt_len == 1024

    return TransformerModelPreLN(
        vocab_size=vocab_size,
        max_src_len=src_len,
        max_tgt_len=tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    )


# ============================================================
# 13. PARAMETER CHECK
# ============================================================

model = build_model(
    V,
    S,
    TT
).to(DEV)

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"\nPARAMETERS: "
    f"{n_params:,} "
    f"({n_params / 1e6:.2f}M)"
)

assert n_params == 129_957_888

print(
    "PASS parameter count = 129,957,888"
)

assert (
    50_000_000
    <= n_params
    <= 300_000_000
)

print(
    "PASS model is inside 50M-300M range"
)


# ============================================================
# 14. DATA
# ============================================================

def mm(name):

    path = f"{TOKD}/{name}.bin"

    assert os.path.exists(path), path

    return np.memmap(
        path,
        dtype=np.uint16,
        mode="r"
    )


data = {
    "acts": mm("acts_train"),
    "sc": mm("sc_train"),
    "tri": mm("tribunals_train_f"),
}

valbins = {
    "acts_val": mm("acts_val"),
    "sc_val": mm("sc_val"),
    "tribunals_val_f": mm("tribunals_val_f"),
}

keys = list(
    CFG["MIX"].keys()
)

probs = np.array(
    list(CFG["MIX"].values()),
    dtype=np.float64
)

probs = probs / probs.sum()


# ============================================================
# 15. WINDOW SPLIT
# ============================================================

def split_window(w):

    w = w[:, :BLOCK]

    src = w[:, :S]

    target = w[:, S:]

    bos = torch.full(
        (
            w.size(0),
            1
        ),
        BOS,
        dtype=w.dtype,
        device=w.device
    )

    tgt_in = torch.cat(
        [
            bos,
            target[:, :-1]
        ],
        dim=1
    )

    return src, tgt_in, target


# ============================================================
# 16. TRAINING BATCH
# ============================================================

def get_batch():

    windows = []

    for _ in range(MB):

        key = np.random.choice(
            keys,
            p=probs
        )

        d = data[key]

        start = np.random.randint(
            0,
            len(d) - BLOCK - 1
        )

        windows.append(
            np.asarray(
                d[
                    start:
                    start + BLOCK
                ],
                dtype=np.int64
            )
        )

    w = torch.from_numpy(
        np.stack(windows)
    ).to(DEV)

    return split_window(w)


# ============================================================
# 17. VALIDATION WINDOWS
# ============================================================

rs = np.random.RandomState(123)

val_windows = {}

for name, d in valbins.items():

    starts = rs.randint(
        0,
        len(d) - BLOCK - 1,
        200
    )

    val_windows[name] = np.stack([
        np.asarray(
            d[
                i:
                i + BLOCK
            ],
            dtype=np.int64
        )

        for i in starts
    ])


# ============================================================
# 18. PREFLIGHT
# ============================================================

@torch.no_grad()
def preflight():

    model.eval()

    src = torch.randint(
        0,
        V,
        (2, S),
        device=DEV
    )

    tin = torch.randint(
        0,
        V,
        (2, TT),
        device=DEV
    )

    with torch.autocast(
        "cuda",
        dtype=torch.float16
    ):

        out = model(
            src,
            tin
        ).float()

    assert out.shape == (
        2,
        TT,
        V
    )

    assert torch.isfinite(
        out
    ).all()

    print(
        "[preflight] shape OK:",
        tuple(out.shape)
    )

    # --------------------------------------------------------
    # Decoder causality
    # --------------------------------------------------------

    t = TT // 2

    tin2 = tin.clone()

    tin2[:, t:] = torch.randint(
        0,
        V,
        tin2[:, t:].shape,
        device=DEV
    )

    with torch.autocast(
        "cuda",
        dtype=torch.float16
    ):

        out2 = model(
            src,
            tin2
        ).float()

    leak = (
        out[:, :t]
        - out2[:, :t]
    ).abs().max().item()

    print(
        f"[preflight] decoder causality: "
        f"{leak:.2e}"
    )

    assert leak < 1e-3

    # --------------------------------------------------------
    # Encoder dependence
    # --------------------------------------------------------

    src2 = torch.randint(
        0,
        V,
        (2, S),
        device=DEV
    )

    with torch.autocast(
        "cuda",
        dtype=torch.float16
    ):

        out3 = model(
            src2,
            tin
        ).float()

    dep = (
        out
        - out3
    ).abs().max().item()

    print(
        f"[preflight] encoder dependence: "
        f"{dep:.2e}"
    )

    assert dep > 0

    # --------------------------------------------------------
    # Initial loss
    # --------------------------------------------------------

    src_b, tin_b, tout_b = get_batch()

    with torch.autocast(
        "cuda",
        dtype=torch.float16
    ):

        lg = model(
            src_b,
            tin_b
        )

    l0 = F.cross_entropy(
        lg.float().reshape(-1, V),
        tout_b.reshape(-1)
    ).item()

    print(
        f"[preflight] initial loss: "
        f"{l0:.4f} "
        f"(uniform={math.log(V):.4f})"
    )

    model.train()


preflight()


# ============================================================
# 19. OPTIMIZER
# ============================================================

try:

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["LR"],
        weight_decay=CFG["WD"],
        fused=True
    )

    print(
        "Optimizer: fused AdamW"
    )

except Exception as e:

    print(
        "WARNING: fused AdamW unavailable"
    )

    print(
        repr(e)
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["LR"],
        weight_decay=CFG["WD"]
    )


# ============================================================
# 20. AMP
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda"
)


# ============================================================
# 21. LR SCHEDULE
# ============================================================

def lr_at(step):

    if step < CFG["WARMUP"]:

        return (
            CFG["LR"]
            * (step + 1)
            / CFG["WARMUP"]
        )

    p = (
        step
        - CFG["WARMUP"]
    ) / max(
        1,
        CFG["STEPS"]
        - CFG["WARMUP"]
    )

    p = min(
        max(
            p,
            0.0
        ),
        1.0
    )

    return CFG["LR"] * (
        CFG["MIN_LR_FRAC"]
        + (
            1
            - CFG["MIN_LR_FRAC"]
        )
        * 0.5
        * (
            1
            + math.cos(
                math.pi * p
            )
        )
    )


# ============================================================
# 22. TRAIN STEP
# ============================================================

def train_step(lr):

    for group in opt.param_groups:

        group["lr"] = lr

    total = 0.0

    for _ in range(GA):

        src, tin, tout = get_batch()

        with torch.autocast(
            "cuda",
            dtype=torch.float16
        ):

            logits = model(
                src,
                tin
            )

            loss = (
                F.cross_entropy(
                    logits.float().reshape(
                        -1,
                        V
                    ),
                    tout.reshape(-1)
                )
                / GA
            )

        scaler.scale(
            loss
        ).backward()

        total += loss.item()

        del (
            src,
            tin,
            tout,
            logits,
            loss
        )

    scaler.unscale_(opt)

    grad_norm = (
        nn.utils.clip_grad_norm_(
            model.parameters(),
            CFG["CLIP"]
        )
        .item()
    )

    scaler.step(
        opt
    )

    scaler.update()

    opt.zero_grad(
        set_to_none=True
    )

    return total, float(
        grad_norm
    )


# ============================================================
# 23. VALIDATION
# ============================================================

@torch.no_grad()
def val_loss(bs=2):

    model.eval()

    results = {}

    for name, windows in val_windows.items():

        total = 0.0
        count = 0

        for i in range(
            0,
            len(windows),
            bs
        ):

            b = torch.from_numpy(
                windows[
                    i:
                    i + bs
                ]
            ).to(DEV)

            src, tin, tout = (
                split_window(b)
            )

            with torch.autocast(
                "cuda",
                dtype=torch.float16
            ):

                logits = model(
                    src,
                    tin
                )

            total += F.cross_entropy(
                logits.float().reshape(
                    -1,
                    V
                ),
                tout.reshape(-1),
                reduction="sum"
            ).item()

            count += tout.numel()

        results[name] = (
            total / count
        )

    model.train()

    return results


# ============================================================
# 24. CHECKPOINT
# ============================================================

def save_ckpt(
    path,
    step,
    best
):

    torch.save(
        {
            "model":
                model.state_dict(),

            "opt":
                opt.state_dict(),

            "scaler":
                scaler.state_dict(),

            "step":
                step,

            "best":
                best,

            "cfg":
                CFG,

            "n_params":
                n_params,
        },
        path
    )




# ============================================================
# EXTRAS FOR EVALUATION / GENERATION SCRIPTS (not part of training)
# ============================================================
dev = DEV                      # some helper scripts use the name `dev`
# Default CKPT below points at the pre-LN run's latest.pt (step 10,000). Set CKPT earlier to override.
CKPT = globals().get("CKPT") or "/kaggle/input/notebooks/achiver7/notebook6cfab69cee/run_arch2_preln_efficient/latest.pt"
if CKPT:
    _ck = torch.load(CKPT, map_location=DEV)
    model.load_state_dict(_ck["model"])
    print("loaded", CKPT, "| step", _ck.get("step"), "| best", _ck.get("best"))
    del _ck
model.eval()
print("definitions ready; model in eval mode")

CUDA: Tesla T4
Config: SRC=768 TGT=256 MB=2 GA=8 effective=16
RUN: /kaggle/working/run_arch2_preln_efficient
FRESH RUN: no previous checkpoint will be loaded.
BOS token: 1

PARAMETERS: 129,957,888 (129.96M)
PASS parameter count = 129,957,888
PASS model is inside 50M-300M range
[preflight] shape OK: (2, 256, 40000)
[preflight] decoder causality: 0.00e+00
[preflight] encoder dependence: 1.54e+00
[preflight] initial loss: 15.2055 (uniform=10.5966)
Optimizer: fused AdamW
loaded /kaggle/input/notebooks/achiver7/notebook6cfab69cee/run_arch2_preln_efficient/latest.pt | step 10000 | best 2.906713534593582
definitions ready; model in eval mode


In [4]:
# ============================================================
# SOURCE-LENGTH CHECK (read-only). The model was trained with a source of exactly SRC_LEN (768) tokens, but
# most test prompts are shorter. This measures how the validation loss on the TARGET tokens changes when the
# encoder is given only the LAST k tokens of the source (a shorter prompt, fed directly, no padding).
# Run in the notebook where the training file defined everything and `model` holds the trained weights
# (load the checkpoint first, e.g. model.load_state_dict(torch.load(PATH)["model"])).
# ============================================================
import numpy as np
import torch
import torch.nn.functional as F

N_WIN = 100
KS = (768, 640, 512, 384)


@torch.no_grad()
def loss_with_src_len(k):
    model.eval()
    out = {}
    for name, w in val_windows.items():
        tot = cnt = 0
        for i in range(N_WIN):
            b = torch.from_numpy(w[i:i + 1]).to(dev)
            src, tin, tout = split_window(b)
            src = src[:, -k:]                              # keep only the last k source tokens
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model(src, tin)
            tot += F.cross_entropy(lg.float().reshape(-1, V), tout.reshape(-1), reduction="sum").item()
            cnt += tout.numel()
        out[name] = tot / cnt
    return out


print(f"validation loss on target tokens, first {N_WIN} windows/source")
base = None
for k in KS:
    r = loss_with_src_len(k)
    m = float(np.mean(list(r.values())))
    base = m if base is None else base
    print(f"source = last {k:>3} tokens | " + " ".join(f"{n}={v:.3f}" for n, v in r.items()) + f" | mean {m:.3f} | change vs 768: {m-base:+.3f}")
print("""
HOW TO READ (explanations, not conclusions):
 * a small change means a shorter prompt (fed directly, no padding) costs little;
 * a large change means the model depends on seeing exactly 768 source tokens - then handling of short prompts needs a decision.
 * test prompts: mean ~609 tokens, max 918 (prompts longer than 768 keep their last 768 tokens).
""")

validation loss on target tokens, first 100 windows/source
source = last 768 tokens | acts_val=2.706 sc_val=3.155 tribunals_val_f=2.934 | mean 2.932 | change vs 768: +0.000
source = last 640 tokens | acts_val=2.729 sc_val=3.179 tribunals_val_f=2.956 | mean 2.955 | change vs 768: +0.023
source = last 512 tokens | acts_val=2.739 sc_val=3.186 tribunals_val_f=2.964 | mean 2.963 | change vs 768: +0.031
source = last 384 tokens | acts_val=2.745 sc_val=3.193 tribunals_val_f=2.977 | mean 2.972 | change vs 768: +0.040

HOW TO READ (explanations, not conclusions):
 * a small change means a shorter prompt (fed directly, no padding) costs little;
 * a large change means the model depends on seeing exactly 768 source tokens - then handling of short prompts needs a decision.
 * test prompts: mean ~609 tokens, max 918 (prompts longer than 768 keep their last 768 tokens).



In [8]:
# ============================================================
# GENERATION + PACKAGING for Architecture 2 (pre-LN encoder-decoder). No training.
# Run AFTER preln_defs_only.py (with CKPT set to the trained checkpoint) in the same notebook.
# Needs: model, tok, BOS, S, TT, DEV, TOKD.
#
# - The encoder runs ONCE per prompt; the decoder runs once per generated token (no KV cache).
# - The encoder sees the LAST min(len(prompt), S) prompt tokens. Prompts shorter than S are fed directly
#   (no padding) - check the effect first with src_length_check.py.
# - YOU choose the decoding settings in GEN; the script refuses to run until they are set.
# ============================================================
import os, json, time, zipfile
import numpy as np
import torch
import torch.nn.functional as F

OUT = "/kaggle/working/submission_arch2"
os.makedirs(OUT, exist_ok=True)

# TEMP=0 -> greedy. TOP_K=0 -> off. TOP_P=1.0 -> off.
GEN = dict(TEMP=0.9, TOP_K=50, TOP_P=1.0, SEED=0)   # same values as the Arch 1 submission 1 (kept for comparability); change if you choose otherwise
missing = [k for k, v in GEN.items() if v is None]
assert not missing, f"set these in GEN first: {missing}"

RANGE = (0, None)            # (0, 5): timing test | (0, None): all | split across accounts, e.g. (0, 500) and (500, None)
PAD_SHORT = False         # False: outputs with < 128 words are reported and block packaging. True: pad by repeating (counted).

test = json.load(open(f"{TOKD}/test.json"))
all_ids = list(test)
a, b = RANGE[0], (RANGE[1] if RANGE[1] is not None else len(all_ids))
ids = all_ids[a:b]
model.eval()
gen = torch.Generator(device=DEV).manual_seed(GEN["SEED"])


def sample(lg):
    if GEN["TEMP"] == 0:
        return lg.argmax(-1, keepdim=True)
    lg = lg / GEN["TEMP"]
    if GEN["TOP_K"]:
        v, _ = torch.topk(lg, min(GEN["TOP_K"], lg.size(-1)))
        lg[lg < v[:, [-1]]] = -float("inf")
    p = F.softmax(lg, -1)
    if GEN["TOP_P"] < 1.0:
        sp, si = torch.sort(p, descending=True)
        cum = sp.cumsum(-1)
        sp[(cum - sp) > GEN["TOP_P"]] = 0
        p = torch.zeros_like(p).scatter_(1, si, sp)
        p = p / p.sum(-1, keepdim=True)
    return torch.multinomial(p, 1, generator=gen)


@torch.no_grad()
def continue_text(prompt):
    prompt_ids = tok.encode(prompt).ids[-S:]
    src = torch.tensor([prompt_ids], device=DEV)
    with torch.autocast("cuda", dtype=torch.float16):
        memory = model.encode(src)
    prefix = torch.tensor([[BOS]], device=DEV)
    new = []
    while len(new) < TT:                                   # prefix length never exceeds TT
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model.decode(prefix, memory)[:, -1, :].float()
        nxt = sample(lg)
        prefix = torch.cat([prefix, nxt], 1)
        new.append(nxt.item())
        if len(new) % 8 == 0 and len(tok.decode(new).split()) >= 128:
            break
    return tok.decode(new).split(), len(prompt_ids)


part = f"{OUT}/pred_part_{a}_{b}.json"
meta_path = f"{OUT}/short_and_meta_{a}_{b}.json"
pred = json.load(open(part)) if os.path.exists(part) else {}
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {"short": {}, "src_tokens": {}}
t0 = time.time()
for n, k in enumerate(ids):
    if k in pred:
        continue
    words, n_src = continue_text(test[k]["text"])
    meta["src_tokens"][k] = n_src
    if len(words) < 128:
        meta["short"][k] = len(words)
        if PAD_SHORT and words:
            words = (words * (128 // len(words) + 1))[:128]
    pred[k] = {"text": " ".join(words[:128])}
    if (n + 1) % 25 == 0 or n + 1 == len(ids):
        json.dump(pred, open(part, "w")); json.dump(meta, open(meta_path, "w"))
        done = len(pred)
        print(f"{done}/{len(ids)} done | {(time.time()-t0)/max(1, done):.1f}s per prompt | short outputs so far: {len(meta['short'])}")
json.dump(pred, open(part, "w")); json.dump(meta, open(meta_path, "w"))

print(f"\nshort outputs (<128 words) in this range: {len(meta['short'])}"
      + (" (padded by repetition)" if PAD_SHORT else " (NOT padded: they block packaging)"))
print("prompt token counts: min", min(meta["src_tokens"].values()), "| max", max(meta["src_tokens"].values()),
      "| shorter than S:", sum(v < S for v in meta["src_tokens"].values()))
assert not any(test[k]["text"][:60] in v["text"] for k, v in pred.items()), "prompt leaked into output"

if set(pred) == set(test):
    bad = [k for k, v in pred.items() if len(v["text"].split()) != 128]
    assert not bad, f"{len(bad)} outputs are not exactly 128 words (e.g. {bad[:3]}); decide how to handle them (PAD_SHORT) before packaging"
    with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(pred, f, ensure_ascii=False)
    with zipfile.ZipFile(f"{OUT}/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")
    print("zip root:", zipfile.ZipFile(f"{OUT}/submission.zip").namelist(), "| IDs:", len(pred))
else:
    print(f"PARTIAL ({len(pred)}/{len(test)} IDs): no zip written. Merge the parts first, then rerun the packaging.")

25/1000 done | 1.3s per prompt | short outputs so far: 0
50/1000 done | 1.3s per prompt | short outputs so far: 0
75/1000 done | 1.3s per prompt | short outputs so far: 0
100/1000 done | 1.3s per prompt | short outputs so far: 0
125/1000 done | 1.3s per prompt | short outputs so far: 0
150/1000 done | 1.3s per prompt | short outputs so far: 0
175/1000 done | 1.3s per prompt | short outputs so far: 1
200/1000 done | 1.3s per prompt | short outputs so far: 1
225/1000 done | 1.3s per prompt | short outputs so far: 1
250/1000 done | 1.3s per prompt | short outputs so far: 1
275/1000 done | 1.3s per prompt | short outputs so far: 1
300/1000 done | 1.3s per prompt | short outputs so far: 1
325/1000 done | 1.3s per prompt | short outputs so far: 1
350/1000 done | 1.3s per prompt | short outputs so far: 1
375/1000 done | 1.3s per prompt | short outputs so far: 1
400/1000 done | 1.3s per prompt | short outputs so far: 1
425/1000 done | 1.3s per prompt | short outputs so far: 1
450/1000 done | 1

AssertionError: 2 outputs are not exactly 128 words (e.g. ['157', '703']); decide how to handle them (PAD_SHORT) before packaging

In [10]:
# ============================================================
# FINISH PACKAGING for Architecture 2 after generate_arch2.py produced all 1000 IDs but a few outputs
# were shorter than 128 words. Run in the SAME kernel (needs: continue_text, test, gen, OUT).
# YOU choose how to handle the short ones (set HANDLE below):
#   "pad"      - repeat the generated words until there are 128 (these outputs are repetitive; counted and recorded)
#   "resample" - regenerate those prompts with other random seeds until >= 128 words (up to 20 tries each)
# ============================================================
import json, zipfile

HANDLE = "resample"                     # <-- "pad" or "resample"
assert HANDLE in ("pad", "resample"), 'set HANDLE to "pad" or "resample"'

part = f"{OUT}/pred_part_0_1000.json"
meta_path = f"{OUT}/short_and_meta_0_1000.json"
pred = json.load(open(part))
meta = json.load(open(meta_path))
assert set(pred) == set(test), "not all 1000 IDs are present"

bad = [k for k, v in pred.items() if len(v["text"].split()) != 128]
print("outputs not exactly 128 words:", {k: len(pred[k]["text"].split()) for k in bad})
print("their prompt token counts   :", {k: meta["src_tokens"].get(k) for k in bad})
for k in bad:
    print(f"\n[{k}] {pred[k]['text'][:300]}")

handled = []
for k in bad:
    if HANDLE == "pad":
        w = pred[k]["text"].split()
        assert w, f"ID {k} has no words at all"
        pred[k]["text"] = " ".join((w * (128 // len(w) + 1))[:128])
        handled.append(k)
    else:
        for t in range(1, 21):
            gen.manual_seed(1000 + t)
            words, _ = continue_text(test[k]["text"])
            if len(words) >= 128:
                pred[k]["text"] = " ".join(words[:128])
                handled.append(k)
                print(f"ID {k}: resampled OK on try {t}")
                break
        else:
            print(f"ID {k}: still short after 20 tries")

still = [k for k, v in pred.items() if len(v["text"].split()) != 128]
assert not still, f"still not 128 words: {still}"
assert not any(test[k]["text"][:60] in v["text"] for k, v in pred.items()), "prompt leaked into output"

meta["handled_short_ids"] = handled
meta["handle_mode"] = HANDLE
json.dump(pred, open(part, "w"))
json.dump(meta, open(meta_path, "w"))

with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
    json.dump(pred, f, ensure_ascii=False)
with zipfile.ZipFile(f"{OUT}/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{OUT}/prediction.json", arcname="prediction.json")
print("\nzip root:", zipfile.ZipFile(f"{OUT}/submission.zip").namelist(), "| IDs:", len(pred),
      "| handled short:", len(handled), f"({HANDLE})")

outputs not exactly 128 words: {'157': 112, '703': 19}
their prompt token counts   : {'157': 594, '703': 605}

[157] the sale deed dated 8th November, 1922, Exhibit C-2, Exhibit C-3, Exhibit C-3, Exhibit C-4, Exhibit D-4, Exhibit D-5, Exhibit C-8, Exhibit C-8, Exhibit D-3, Exhibit D- 11, Exhibit B-1, Exhibit C-11, Exhibit D-5, Exhibit C-9, Exhibit C-29, Exhibit C-9, Exhibit C-11 and Exhibit D-5 to Exhibit C-11, Ex

[703] (1) the charge of murder was concerned with the offence of murder. It was then held:- "...... ".. "..
ID 157: resampled OK on try 1
ID 703: resampled OK on try 1

zip root: ['prediction.json'] | IDs: 1000 | handled short: 2 (resample)


In [7]:
   for k in ids[:2]:
       print(pred[k]["text"][:500], "\n---")

case. 6. The State of Bihar had no objection to the maintainability of this petition. In Union of India & Ors. Vs. M/s. Union of India & Ors., (2006) 24 SCC 241, this Court while dealing with the scope of the writ petition, observed as under: "...The High Court was dealing with the provisions of law and not by any provision of law, for the time being in force, is not confined to proceedings for acquisition of land, but in those circumstances, it is open to the State to bring the matter in the la 
---
offences with a view to protect the accused, which is not the only cause of action, which is the subject matter. After investigation, the learned Magistrate vide order dated 26.07.2009 has issued a direction to the Magistrate and the Magistrate for taking cognizance of the offence committed committed an offence punishable under Sections 376 and 201 of the Ranbir Penal Code. As there is no delay in lodging the complaint before the Judicial Magistrate he cannot be denied the benefit of doubt

In [11]:
# ============================================================
# DATA AUDIT (read-only, no training). Run in the notebook after the training/defs file has defined:
#   data (dict name -> memmap of token ids), tok, TOKD (folder with test.json), BLOCK, and the training config.
# Answers three questions with numbers (no conclusions):
#   1. How many tokens does each source have, and how many passes (epochs) did your run make over it?
#   2. How many tokens per <eos> (document length) does each source have?
#   3. How many test prompts end with a token sequence that also appears verbatim in the training files?
# ============================================================
import json
import numpy as np

# ---- what your run used: edit only if these differ from your config ----
cfg = globals().get("CFG", {})
STEPS      = cfg.get("STEPS", 10000)
TOK_PER_STEP = cfg.get("MICRO_BATCH", 2) * cfg.get("GRAD_ACCUM", 8) * BLOCK     # window tokens per optimizer step
MIX        = cfg.get("MIX") or globals().get("MIX")                             # dict: source -> sampling weight
EOS        = tok.token_to_id("<eos>")

total_seen = STEPS * TOK_PER_STEP
print(f"window tokens per step {TOK_PER_STEP:,} | steps {STEPS:,} | total window tokens seen {total_seen:,}\n")

# ---------------- 1 and 2: size, passes, document length ----------------
print(f"{'source':8s} {'tokens':>14s} {'mix w':>7s} {'tokens drawn':>14s} {'passes':>8s} {'tok/<eos>':>10s}")
for name, d in data.items():
    n = len(d)
    w = (MIX or {}).get(name)
    drawn = total_seen * w if w is not None else float("nan")
    # count <eos> on a 50M-token sample (the whole file would be slow for big memmaps)
    m = min(n, 50_000_000)
    n_eos = int((np.asarray(d[:m]) == EOS).sum())
    print(f"{name:8s} {n:14,d} {('%.3f' % w) if w is not None else '  n/a':>7s} {drawn:14,.0f} {drawn / n:8.2f} {m / max(n_eos, 1):10.0f}")
print("""
HOW TO READ (explanations, not conclusions):
 * passes < 1 : the run saw only part of that source once, so the corpus is not the limit; more steps would still see new text.
 * passes > 1 : text repeats; extra steps give diminishing returns and may overfit.
 * tok/<eos>  : average document length in tokens; windows of 1024 tokens contain an <eos> more often when documents are short.
""")

# ---------------- 3: do test-prompt endings appear verbatim in the training files? ----------------
K = 16                                              # n-gram length in tokens
test = json.load(open(f"{TOKD}/test.json"))
P = np.uint64(1000003)

def poly_hash_rows(a, k):
    """hash of every length-k window of 1-D array a (uint64 polynomial hash, wraparound is fine)."""
    a = a.astype(np.uint64)
    h = np.zeros(len(a) - k + 1, dtype=np.uint64)
    for j in range(k):
        h = h * P + a[j:len(a) - k + 1 + j]
    return h

tails = {}
for key, v in test.items():
    ids = tok.encode(v["text"]).ids
    if len(ids) >= K:
        tails[key] = poly_hash_rows(np.array(ids[-K:]), K)[0]
tail_vals = np.array(list(tails.values()), dtype=np.uint64)
print(f"test prompts with >= {K} tokens: {len(tails)} / {len(test)}")

found = set()
CH = 20_000_000
for name, d in data.items():
    for s in range(0, len(d) - K, CH):
        chunk = np.asarray(d[s:s + CH + K - 1])
        if len(chunk) < K:
            continue
        hits = np.isin(poly_hash_rows(chunk, K), tail_vals)
        if hits.any():
            hv = set(poly_hash_rows(chunk, K)[hits].tolist())
            found |= {k for k, h in tails.items() if int(h) in hv}
    print(f"after {name:6s}: prompts whose last {K} tokens appear in training data so far: {len(found)}")
print(f"\nTOTAL: {len(found)} of {len(tails)} test prompts end with a {K}-token sequence found verbatim in the training files.")
print("""
HOW TO READ (explanations, not conclusions):
 * a large count means test documents (or near-copies) are inside the training corpus; a model that memorises more of the corpus
   would then score higher on overlap metrics. Check the course rules before relying on this in any way.
 * a count near zero means test prompts are unseen text, so overlap metrics cannot come from memorising them.
 * a 16-token match can be boilerplate (a standard legal clause), so a few matches are expected even for unseen documents.
""")

window tokens per step 16,384 | steps 10,000 | total window tokens seen 163,840,000

source           tokens   mix w   tokens drawn   passes  tok/<eos>
acts        145,554,561   0.273     44,728,320     0.31       8054
sc          249,782,556   0.469     76,840,960     0.31       4744
tri         137,567,074   0.258     42,270,720     0.31       1449

HOW TO READ (explanations, not conclusions):
 * passes < 1 : the run saw only part of that source once, so the corpus is not the limit; more steps would still see new text.
 * passes > 1 : text repeats; extra steps give diminishing returns and may overfit.
 * tok/<eos>  : average document length in tokens; windows of 1024 tokens contain an <eos> more often when documents are short.

test prompts with >= 16 tokens: 1000 / 1000
after acts  : prompts whose last 16 tokens appear in training data so far: 24
after sc    : prompts whose last 16 tokens appear in training data so far: 175
after tri   : prompts whose last 16 tokens appear in traini